# Taller de movilidad urbana: taxis y clima en Nueva York

**Caso de estudio:** viajes de NYC TLC Yellow Taxi durante enero de 2024.

En este taller construiremos un flujo reproducible: adquisición, inspección, auditoría de calidad, enriquecimiento por zonas, agregación temporal, integración con clima y análisis exploratorio. No buscamos demostrar causalidad, sino aprender a formular y comprobar preguntas con datos reales.

## Objetivos

Al finalizar podrás:

- construir y validar una URL de datos abiertos;
- inspeccionar estructura, tipos y faltantes antes de analizar;
- convertir reglas de calidad en una tabla de auditoría;
- enriquecer viajes mediante uniones `many_to_one`;
- agregar pickups por zona y hora sin sesgar la serie temporal;
- convertir observaciones meteorológicas de UTC a `America/New_York`;
- comunicar patrones y límites de un análisis exploratorio.

> **Pregunta guía:** ¿cómo varían los pickups de Yellow Taxi por lugar y hora, y qué relación exploratoria muestran con temperatura y precipitación?

## Ficha de fuentes

| Fuente | Recurso usado | Papel en el taller |
|---|---|---|
| [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) | Yellow Taxi, enero de 2024 (`yellow_tripdata_2024-01.parquet`) | fecha/hora, zonas, distancia, pasajeros e importes de viajes reportados |
| NYC TLC | [Taxi Zone Lookup](https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv) | traduce `LocationID` a borough, zona y `service_zone` |
| NYC TLC | [Taxi Zone Shapefile](https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip) | geometrías para la extensión espacial opcional |
| [NOAA GHCNh](https://www.ncei.noaa.gov/products/global-historical-climatology-network-hourly) | estación `USW00094728`, año 2024 | observaciones de clima con marcas temporales UTC |

**Periodo común:** enero de 2024 en hora local de Nueva York. Los archivos se leen desde sus URL o desde un directorio temporal; este notebook no guarda datasets en el repositorio. Consulta la documentación y licencias de cada proveedor antes de reutilizar los datos.

## 1. Preparación y modo de ejecución

En Colab normalmente basta con ejecutar los imports. Si falta el motor Parquet, descomenta la instalación. La extensión espacial se instala solo si se desea ejecutar esa sección.

In [ ]:
# Instalación opcional para Google Colab/Jupyter (descomentar si hace falta):
# %pip install -q pyarrow
# %pip install -q geopandas folium requests

from io import StringIO
import tempfile
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# True: primera semana de enero; False: todo enero. No hay muestreo aleatorio.
MODO_CLASE = False
INICIO_MES = pd.Timestamp("2024-01-01")
FIN_MES = pd.Timestamp("2024-02-01")
FIN_ANALISIS = pd.Timestamp("2024-01-08") if MODO_CLASE else FIN_MES
ZONA_HORARIA = "America/New_York"

print(f"Ventana: {INICIO_MES} hasta {FIN_ANALISIS} (fin no incluido)")

`MODO_CLASE` limita la lectura mediante un filtro temporal de Parquet cuando el motor lo permite. La selección es una semana completa y determinista, no una muestra aleatoria: así preservamos horas consecutivas y el flujo de agregación. En modo completo, la misma lógica procesa todo enero.

## 2. URL validada y carga reproducible

In [ ]:
def construir_url_tlc(tipo, anio, mes):
    """Construye una URL TLC válida para un tipo, año y mes."""
    prefijos = {
        "yellow": "yellow_tripdata",
        "green": "green_tripdata",
        "fhv": "fhv_tripdata",
        "fhvhv": "fhvhv_tripdata",
    }
    if tipo not in prefijos:
        opciones = ", ".join(prefijos)
        raise ValueError(f"tipo debe ser uno de: {opciones}")
    if isinstance(anio, bool) or not isinstance(anio, (int, np.integer)) or not 2009 <= anio <= 2100:
        raise ValueError("anio debe ser un entero entre 2009 y 2100")
    if isinstance(mes, bool) or not isinstance(mes, (int, np.integer)) or not 1 <= mes <= 12:
        raise ValueError("mes debe ser un entero entre 1 y 12")
    archivo = f"{prefijos[tipo]}_{anio}-{mes:02d}.parquet"
    return f"https://d37ci6vzurychx.cloudfront.net/trip-data/{archivo}"

URL_VIAJES = construir_url_tlc("yellow", 2024, 1)
URL_ZONAS = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
URL_GEOMETRIAS = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"
URL_CLIMA = (
    "https://www.ncei.noaa.gov/oa/global-historical-climatology-network/"
    "hourly/access/by-year/2024/psv/GHCNh_USW00094728_2024.psv"
)
print(URL_VIAJES)

In [ ]:
COLUMNAS_VIAJES = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "PULocationID",
    "DOLocationID", "fare_amount", "total_amount",
]
filtros = [
    ("tpep_pickup_datetime", ">=", INICIO_MES.to_pydatetime()),
    ("tpep_pickup_datetime", "<", FIN_ANALISIS.to_pydatetime()),
]

# PyArrow aplica el filtro por fecha al leer los grupos de filas compatibles.
viajes_originales = pd.read_parquet(
    URL_VIAJES, columns=COLUMNAS_VIAJES, filters=filtros, engine="pyarrow"
)
print(f"Viajes cargados: {len(viajes_originales):,}")

### Pregunta breve

¿Por qué una semana consecutiva es preferible a una muestra aleatoria si queremos comparar horas y días? Escribe una hipótesis antes de continuar.

<details><summary>Ver respuesta orientativa</summary>Una muestra aleatoria puede dejar horas incompletas y alterar sus conteos. Una ventana consecutiva conserva el orden, los ciclos diarios y el denominador temporal, aunque no necesariamente representa el mes completo.</details>

## 3. Inspección antes de limpiar

No debemos decidir reglas sin conocer el esquema. Observa dimensiones, ejemplos, tipos y faltantes.

In [ ]:
print("Shape:", viajes_originales.shape)
display(viajes_originales.head())
print("\nInfo:")
viajes_originales.info()
print("\nTipos:")
display(viajes_originales.dtypes.rename("dtype").to_frame())

faltantes = (
    viajes_originales.isna().sum().rename("n_faltantes").to_frame()
    .assign(porcentaje=lambda x: 100 * x["n_faltantes"] / len(viajes_originales))
    .sort_values("porcentaje", ascending=False)
)
display(faltantes)

**Observa:** `passenger_count` puede faltar por la forma de reporte. Su ausencia se conserva y no invalida automáticamente un viaje. Solo marcaremos como problemáticos los valores presentes fuera de un intervalo operativo razonable.

## 4. Calidad y auditoría

Las reglas siguientes son criterios analíticos, no verdades universales. Distancia máxima de 100 millas, hasta 8 pasajeros e importes menores a USD 1,000 son umbrales conservadores para detectar registros extremos o inválidos. Cada regla queda en una columna booleana.

In [ ]:
calidad = viajes_originales.copy()
calidad["duracion_min"] = (
    calidad["tpep_dropoff_datetime"] - calidad["tpep_pickup_datetime"]
).dt.total_seconds() / 60

calidad["ok_pickup"] = calidad["tpep_pickup_datetime"].notna() & calidad["tpep_pickup_datetime"].between(INICIO_MES, FIN_ANALISIS, inclusive="left")
calidad["ok_dropoff"] = calidad["tpep_dropoff_datetime"].notna()
calidad["ok_duracion"] = calidad["duracion_min"].gt(0) & calidad["duracion_min"].le(24 * 60)
calidad["ok_distancia"] = calidad["trip_distance"].between(0, 100, inclusive="both")
calidad["ok_id_pickup_presente"] = calidad["PULocationID"].notna()
calidad["ok_id_dropoff_presente"] = calidad["DOLocationID"].notna()
calidad["ok_importes"] = (
    calidad["fare_amount"].between(0, 1_000, inclusive="both")
    & calidad["total_amount"].between(0, 1_000, inclusive="both")
    & calidad["total_amount"].ge(calidad["fare_amount"])
)
calidad["ok_pasajeros"] = calidad["passenger_count"].isna() | calidad["passenger_count"].between(0, 8, inclusive="both")

REGLAS = [c for c in calidad.columns if c.startswith("ok_")]
calidad["registro_valido"] = calidad[REGLAS].all(axis=1)
auditoria = pd.DataFrame({
    "regla": REGLAS,
    "cumplen": [int(calidad[c].sum()) for c in REGLAS],
    "incumplen": [int((~calidad[c]).sum()) for c in REGLAS],
})
auditoria["porcentaje_incumple"] = 100 * auditoria["incumplen"] / len(calidad)
display(auditoria.sort_values("porcentaje_incumple", ascending=False))

In [ ]:
# Se mantienen por separado para no borrar silenciosamente información.
viajes_rechazados = calidad.loc[~calidad["registro_valido"]].copy()
viajes_validos = calidad.loc[calidad["registro_valido"]].copy()
print(f"Válidos: {len(viajes_validos):,} | Rechazados: {len(viajes_rechazados):,}")
display(viajes_rechazados[COLUMNAS_VIAJES + ["duracion_min"] + REGLAS].head())

### Ejercicio 1: sensibilidad de una regla

Calcula cuántos viajes quedarían fuera si la distancia máxima fuese 50 millas, sin modificar `viajes_validos`.

In [ ]:
fuera_con_umbral_50 = (~calidad['trip_distance'].between(0, 50)).sum()
print(f'Viajes fuera con umbral de 50 millas: {fuera_con_umbral_50}')


<details><summary>Ver solución</summary>

```python
fuera_con_umbral_50 = (~calidad["trip_distance"].between(0, 50)).sum()
print(fuera_con_umbral_50)
```

Conviene comparar este resultado con `ok_distancia` y justificar el umbral con conocimiento del dominio.
</details>

## 5. Enriquecimiento con zonas

### Objetivo

La tabla de viajes identifica el origen mediante `PULocationID` y el destino mediante `DOLocationID`. Estos códigos son útiles para relacionar tablas, pero no permiten interpretar directamente dónde comenzó o terminó un viaje.

En esta sección agregaremos a cada viaje el nombre de la zona, el borough y la clasificación de servicio tanto del origen como del destino. Conceptualmente:

```text
PULocationID = 161
        ↓
pickup_zone = Midtown Center
pickup_borough = Manhattan
pickup_service_zone = Yellow Zone
```

El enriquecimiento no cambia la unidad de observación: cada fila continúa representando un viaje Yellow Taxi reportado. Solo agrega contexto geográfico a sus identificadores.

### 5.1. El catálogo de zonas

`taxi_zone_lookup.csv` es un **catálogo de correspondencias** publicado por NYC TLC. No contiene viajes: contiene una descripción por `LocationID`.

| Columna | Significado |
|---|---|
| `LocationID` | Identificador numérico de la zona TLC |
| `Borough` | Distrito amplio, como Manhattan, Queens o Brooklyn |
| `Zone` | Nombre específico de la zona TLC |
| `service_zone` | Clasificación operativa, como `Yellow Zone`, `Boro Zone` o `Airports` |

Para que el catálogo pueda representar el lado **uno** de una unión, `LocationID` debe ser único.

In [ ]:
zonas = pd.read_csv(URL_ZONAS)
display(zonas.head())
print("Cantidad de zonas:", len(zonas))
print("LocationID es único:", zonas["LocationID"].is_unique)

if zonas["LocationID"].duplicated().any():
    raise ValueError("Taxi Zone Lookup contiene LocationID duplicados")

### 5.2. Un catálogo, dos roles

Cada viaje consulta el mismo catálogo desde dos papeles diferentes:

```text
PULocationID → zona de origen o pickup
DOLocationID → zona de destino o dropoff
```

Por eso creamos `zonas_pickup` y `zonas_dropoff`. No son dos catálogos distintos ni duplican los viajes: son dos versiones del mismo catálogo con nombres que explicitan el rol de sus columnas.

| Origen | Destino |
|---|---|
| `pickup_zone` | `dropoff_zone` |
| `pickup_borough` | `dropoff_borough` |
| `pickup_service_zone` | `dropoff_service_zone` |

Renombrar antes de unir evita nombres ambiguos como `Zone_x` y `Zone_y`. `DataFrame.rename()` devuelve estas versiones adaptadas sin modificar `zonas`.

In [ ]:
zonas_pickup = zonas.rename(columns={
    "LocationID": "PULocationID", "Borough": "pickup_borough",
    "Zone": "pickup_zone", "service_zone": "pickup_service_zone",
})
zonas_dropoff = zonas.rename(columns={
    "LocationID": "DOLocationID", "Borough": "dropoff_borough",
    "Zone": "dropoff_zone", "service_zone": "dropoff_service_zone",
})

display(zonas_pickup.head(2))
display(zonas_dropoff.head(2))

### 5.3. Cardinalidad `many_to_one`

La relación esperada es **muchos viajes → una descripción de zona**. Un mismo `PULocationID` o `DOLocationID` puede aparecer en miles de viajes, pero cada identificador debe aparecer como máximo una vez en el catálogo correspondiente.

`validate="many_to_one"` hace que pandas compruebe esta condición durante la unión. Si el catálogo tuviera dos filas para `LocationID = 161`, tres viajes con ese identificador producirían seis filas:

```text
3 viajes × 2 coincidencias en el catálogo = 6 filas
```

Esa multiplicación inflaría conteos, importes, promedios y mapas. La validación detiene el proceso con un error en lugar de producir silenciosamente un resultado incorrecto. También ayuda a detectar catálogos repetidos, mezcla de versiones o descripciones contradictorias para un mismo ID.

### 5.4. Dos uniones izquierdas

Primero agregamos los atributos del origen y después los del destino. `how="left"` conserva todos los viajes válidos. Si un ID no aparece en el catálogo, el viaje permanece y sus atributos geográficos quedan como `NaN`, lo que permite auditar la falta de correspondencia en vez de eliminarla silenciosamente.

In [ ]:
viajes_zonas = (
    viajes_validos.merge(zonas_pickup, on="PULocationID", how="left", validate="many_to_one")
    .merge(zonas_dropoff, on="DOLocationID", how="left", validate="many_to_one")
)

### 5.5. Controles posteriores e interpretación

Después de unir comprobamos dos propiedades diferentes:

1. **Conservación de filas:** la cantidad de viajes no debe cambiar.
2. **Cobertura del catálogo:** contamos los viajes cuyo ID no encontró nombre de zona.

`many_to_one` no garantiza cobertura completa ni detecta viajes duplicados, IDs faltantes, nombres incorrectos en una fila única o el uso de la clave equivocada. Por eso se complementa con estos controles y con la revisión semántica de las columnas obtenidas.

In [ ]:
assert len(viajes_zonas) == len(viajes_validos), "La unión cambió el número de viajes"

sin_nombre_pickup = viajes_zonas["pickup_zone"].isna().sum()
sin_nombre_dropoff = viajes_zonas["dropoff_zone"].isna().sum()
print("Viajes antes y después:", len(viajes_validos), len(viajes_zonas))
print("Sin zona pickup:", sin_nombre_pickup, "| Sin zona dropoff:", sin_nombre_dropoff)

COLUMNAS_ZONA = [
    "PULocationID", "pickup_borough", "pickup_zone", "pickup_service_zone",
    "DOLocationID", "dropoff_borough", "dropoff_zone", "dropoff_service_zone",
]
display(viajes_zonas[COLUMNAS_ZONA].head())

### Resultado de la sección

Cada fila sigue representando un viaje, pero ahora permite interpretar tanto su origen como su destino. Las zonas son áreas geográficas agregadas definidas por TLC: no indican una dirección ni una coordenada exacta, y el enriquecimiento no agrega demanda total, solicitudes no atendidas ni vehículos disponibles.

### Pregunta breve

¿Qué ocurriría si `LocationID = 161` apareciera dos veces en el catálogo y no utilizáramos `validate="many_to_one"`?

<details><summary>Ver respuesta orientativa</summary>Cada viaje con ese identificador encontraría dos coincidencias y aparecería dos veces en el resultado. Los conteos y agregaciones posteriores quedarían inflados. Con `validate="many_to_one"`, pandas detecta que el lado del catálogo no es único y detiene la unión.</details>

## 6. Agregación zona-hora de pickups

Los timestamps TLC se interpretan como hora local. En enero de 2024 no hay transición de horario de verano, pero explicitar la zona horaria evita una unión ambigua con NOAA.

In [ ]:
viajes_zonas["pickup_hora"] = (
    viajes_zonas["tpep_pickup_datetime"]
    .dt.tz_localize(ZONA_HORARIA, ambiguous="raise", nonexistent="raise")
    .dt.floor("h")
)
pickups_zona_hora = (
    viajes_zonas.groupby(
        ["pickup_hora", "PULocationID", "pickup_borough", "pickup_zone"],
        observed=True, dropna=False,
    )
    .size().rename("pickups").reset_index()
)
assert pickups_zona_hora.duplicated(["pickup_hora", "PULocationID"]).sum() == 0
display(pickups_zona_hora.head())

## 7. Clima: UTC, hora local y resumen horario

GHCNh puede contener varias observaciones dentro de una hora. Seleccionaremos variables numéricas, convertiremos valores no numéricos a faltantes y resumiremos a una fila por hora. Temperatura usa media; precipitación usa suma con `min_count=1` para no convertir una hora totalmente faltante en cero.

In [ ]:
COLUMNAS_CLIMA = [
    "STATION", "Station_name", "DATE", "temperature",
    "relative_humidity", "wind_speed", "precipitation", "visibility",
]
with urlopen(URL_CLIMA) as respuesta:
    texto_clima = respuesta.read().decode("utf-8")
clima_bruto = pd.read_csv(StringIO(texto_clima), sep="|", usecols=COLUMNAS_CLIMA, low_memory=False)
assert clima_bruto["STATION"].astype(str).eq("USW00094728").all()
print("Observaciones meteorológicas cargadas:", len(clima_bruto))

In [ ]:
clima = clima_bruto.copy()
clima["fecha_utc"] = pd.to_datetime(clima["DATE"], utc=True, errors="coerce")
clima["fecha_ny"] = clima["fecha_utc"].dt.tz_convert(ZONA_HORARIA)
inicio_local = INICIO_MES.tz_localize(ZONA_HORARIA)
fin_local = FIN_ANALISIS.tz_localize(ZONA_HORARIA)
clima = clima.loc[clima["fecha_ny"].between(inicio_local, fin_local, inclusive="left")].copy()

VARIABLES_CLIMA = ["temperature", "relative_humidity", "wind_speed", "precipitation", "visibility"]
clima[VARIABLES_CLIMA] = clima[VARIABLES_CLIMA].apply(pd.to_numeric, errors="coerce")
clima["hora"] = clima["fecha_ny"].dt.floor("h")
clima_horario = (
    clima.groupby("hora", as_index=False)
    .agg(
        temperatura_c=("temperature", "mean"),
        humedad_relativa=("relative_humidity", "mean"),
        viento=("wind_speed", "mean"),
        precipitacion_mm=("precipitation", lambda s: s.sum(min_count=1)),
        visibilidad=("visibility", "mean"),
        observaciones=("fecha_ny", "size"),
    )
)
if clima_horario["hora"].duplicated().any():
    raise ValueError("El resumen climático no es único por hora")
display(clima_horario.head())

### Ejercicio 2: cobertura temporal

Compara las horas esperadas de la ventana con las horas observadas en `clima_horario`. ¿Hay huecos?

In [ ]:
horas_esperadas = pd.date_range(inicio_local, fin_local, freq='h', inclusive='left')
horas_sin_clima = horas_esperadas.difference(clima_horario['hora'])
print('Horas esperadas:', len(horas_esperadas))
print('Horas sin observacion:', len(horas_sin_clima))


<details><summary>Ver solución</summary>

```python
horas_esperadas = pd.date_range(inicio_local, fin_local, freq="h", inclusive="left")
horas_sin_clima = horas_esperadas.difference(clima_horario["hora"])
print("Horas esperadas:", len(horas_esperadas))
print("Horas sin observación:", len(horas_sin_clima))
display(horas_sin_clima[:10])
```
</details>

## 8. Unión zona-hora con clima

Cada fila zona-hora debe encontrar como máximo una fila climática. La unión izquierda conserva los pickups aunque el clima falte.

In [ ]:
zona_hora_clima = pickups_zona_hora.merge(
    clima_horario, left_on="pickup_hora", right_on="hora",
    how="left", validate="many_to_one", indicator=True,
)
assert len(zona_hora_clima) == len(pickups_zona_hora), "La unión alteró la cardinalidad izquierda"
auditoria_union = zona_hora_clima["_merge"].value_counts(dropna=False).rename_axis("resultado").to_frame("filas")
display(auditoria_union)
zona_hora_clima = zona_hora_clima.drop(columns="_merge")

### ¿Por qué el clima queda repetido por zona?

La unidad de `zona_hora_clima` es **una zona en una hora**. En cambio, `clima_horario` contiene una sola observación por hora. Cuando esa observación se une con varias zonas de la misma hora, pandas la copia en cada fila zona-hora.

Ejemplo conceptual:

| zona | hora | pickups | temperatura |
|---|---|---:|---:|
| A | 08:00 | 20 | 5 °C |
| B | 08:00 | 35 | 5 °C |
| C | 08:00 | 15 | 5 °C |

Los tres valores de `5 °C` no son mediciones diferentes: son la misma observación meteorológica replicada por la unión `many_to_one`. Esta repetición es intencional, no un duplicado accidental. Además, NOAA representa una estación meteorológica; no estamos estimando un clima diferente para cada zona.

## 9. Análisis exploratorio con pandas y matplotlib

Lee cada gráfico como evidencia descriptiva. Pregunta siempre: ¿qué unidad representa?, ¿qué filtros se aplicaron?, ¿qué explicación alternativa existe?

In [ ]:
viajes_por_hora = viajes_zonas.set_index("pickup_hora").resample("h").size()
ax = viajes_por_hora.plot(figsize=(12, 4), color="#1f77b4", linewidth=1.5)
ax.set(title="Viajes por hora", xlabel="Hora local", ylabel="Viajes")
plt.show()

In [ ]:
top_zonas = viajes_zonas["pickup_zone"].value_counts().head(12).sort_values()
ax = top_zonas.plot.barh(figsize=(9, 5), color="#d95f02")
ax.set(title="Zonas con más pickups", xlabel="Viajes", ylabel="Zona de pickup")
plt.show()
display(top_zonas.sort_values(ascending=False).rename("pickups").to_frame())

In [ ]:
patron = viajes_zonas.assign(
    dia=viajes_zonas["pickup_hora"].dt.day_name(),
    hora_dia=viajes_zonas["pickup_hora"].dt.hour,
).pivot_table(index="dia", columns="hora_dia", values="PULocationID", aggfunc="size", fill_value=0)
orden_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
patron = patron.reindex([d for d in orden_dias if d in patron.index])
fig, ax = plt.subplots(figsize=(12, 4))
imagen = ax.imshow(patron, aspect="auto", cmap="YlOrRd")
ax.set(title="Mapa de calor de pickups por día y hora", xlabel="Hora", ylabel="Día")
ax.set_xticks(range(24), labels=range(24))
ax.set_yticks(range(len(patron.index)), labels=patron.index)
fig.colorbar(imagen, ax=ax, label="Pickups")
plt.show()

In [ ]:
limite_99 = viajes_validos["duracion_min"].quantile(0.99)
ax = viajes_validos.loc[viajes_validos["duracion_min"] <= limite_99, "duracion_min"].plot.hist(
    bins=50, figsize=(9, 4), color="#7570b3", edgecolor="white"
)
ax.set(title="Distribución de duración (hasta percentil 99)", xlabel="Minutos", ylabel="Viajes")
plt.show()
print("Percentil 99 mostrado:", round(limite_99, 1), "minutos")

### 9.1. De zona-hora a una observación por hora

Para comparar la actividad total de taxis con el clima debemos volver de la unidad **zona-hora** a la unidad **hora**. En el ejemplo anterior:

```text
pickups = 20 + 35 + 15 = 70
temperatura = 5 °C
```

Por eso usamos operaciones diferentes:

- `sum` para `pickups`, porque cada zona aporta viajes distintos;
- `first` para temperatura y precipitación, porque cada fila contiene una copia del mismo clima horario.

Sumar la temperatura produciría `5 + 5 + 5 = 15 °C`, un valor sin interpretación física. El uso de `first` requiere comprobar antes que una misma hora no contenga valores climáticos diferentes entre zonas.

In [ ]:
# Verificamos el supuesto necesario para conservar una sola copia con 'first'.
variables_clima_repetidas = ["temperatura_c", "precipitacion_mm"]
consistencia_clima = (
    zona_hora_clima
    .groupby("pickup_hora")[variables_clima_repetidas]
    .nunique(dropna=False)
)
assert consistencia_clima.le(1).all().all(), (
    "Una misma hora contiene valores climáticos diferentes entre zonas"
)
print("Consistencia climática por hora: OK")

# El clima está repetido por zona; volvemos a una observación por hora.
comparacion_horaria = (
    zona_hora_clima.groupby("pickup_hora", as_index=False)
    .agg(
        pickups=("pickups", "sum"),
        temperatura_c=("temperatura_c", "first"),
        precipitacion_mm=("precipitacion_mm", "first"),
    )
)
assert not comparacion_horaria["pickup_hora"].duplicated().any()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(comparacion_horaria["temperatura_c"], comparacion_horaria["pickups"], alpha=0.65, color="#1b9e77")
axes[0].set(xlabel="Temperatura (°C)", ylabel="Pickups por hora", title="Pickups y temperatura")
axes[1].scatter(comparacion_horaria["precipitacion_mm"], comparacion_horaria["pickups"], alpha=0.65, color="#377eb8")
axes[1].set(xlabel="Precipitación horaria (mm)", ylabel="Pickups por hora", title="Pickups y precipitación")
plt.tight_layout()
plt.show()
display(comparacion_horaria[["pickups", "temperatura_c", "precipitacion_mm"]].corr().round(3))

### Pregunta breve

¿Qué ocurriría si sumáramos la temperatura después de haberla replicado por zona?

<details><summary>Ver respuesta orientativa</summary>La misma observación se contabilizaría varias veces y produciría una temperatura sin interpretación física. Los pickups sí se suman porque cada zona aporta viajes diferentes; el clima se conserva una sola vez porque es el mismo valor replicado.</details>

### Ejercicio 3: interpretar sin exagerar

**Tu respuesta:**

1. Se observa una correlacion positiva baja entre la temperatura horaria y los pickups agregados por hora.
2. La gran mayoria de las horas tienen 0 mm de lluvia; en los momentos de lluvia los pickups muestran una dispersion similar a las horas secas.
3. Explicacion alternativa: Las variaciones de demanda responden al ciclo diario laboral y al dia de la semana, que coinciden con las curvas termicas diurnas.


## 10. Exportar el producto fuera del repositorio

Antes de exportar comprobamos la clave, ordenamos el producto y usamos el directorio temporal del sistema. El nombre distingue el modo de clase del mes completo para no declarar una cobertura que no fue procesada.

In [ ]:
producto = zona_hora_clima.sort_values(["pickup_hora", "PULocationID"]).copy()
if producto.duplicated(["pickup_hora", "PULocationID"]).any():
    raise ValueError("La clave zona-hora no es única")

alcance = "primera_semana_enero_2024" if MODO_CLASE else "enero_2024"
ruta_producto = Path(tempfile.gettempdir()) / f"zona_hora_{alcance}.parquet"
producto.to_parquet(ruta_producto, index=False)
print("Producto temporal:", ruta_producto)
print("Unidad: zona de origen-hora local | Zona horaria:", ZONA_HORARIA)

## 11. Extensión espacial opcional

Esta celda descarga el ZIP del shapefile a un directorio temporal, extrae juntos sus componentes (`.shp`, `.dbf`, `.shx`, `.prj`) y construye un mapa Folium. La extracción explícita evita depender de que el backend geoespacial admita rutas virtuales `zip://`; al finalizar, el directorio temporal se elimina y no deja geometrías en el repositorio. Si `geopandas`, `folium` o `requests` no están instalados, informa exactamente cómo habilitar la sección y el resto del taller sigue siendo utilizable.

In [ ]:
try:
    import folium
    import geopandas as gpd
    import requests
    import zipfile
except ModuleNotFoundError as error:
    print(
        f"Sección espacial omitida: falta '{error.name}'. "
        "Descomenta `%pip install -q geopandas folium requests`, reinicia si es necesario y vuelve a ejecutar."
    )
else:
    with tempfile.TemporaryDirectory() as directorio_base:
        ruta_zip = Path(directorio_base) / "taxi_zones.zip"
        respuesta = requests.get(URL_GEOMETRIAS, timeout=60)
        respuesta.raise_for_status()
        ruta_zip.write_bytes(respuesta.content)
        if not zipfile.is_zipfile(ruta_zip):
            raise zipfile.BadZipFile("La descarga de zonas TLC no es un archivo ZIP válido")

        directorio_extraido = Path(directorio_base) / "taxi_zones_extraido"
        directorio_extraido.mkdir()
        with zipfile.ZipFile(ruta_zip) as archivo_zip:
            archivo_zip.extractall(directorio_extraido)

        archivos_shp = sorted(directorio_extraido.rglob("*.shp"))
        if not archivos_shp:
            raise FileNotFoundError("No se encontró un archivo .shp dentro del ZIP de zonas TLC")
        ruta_shp = next((ruta for ruta in archivos_shp if ruta.name == "taxi_zones.shp"), archivos_shp[0])
        geo_zonas = gpd.read_file(ruta_shp).to_crs(epsg=4326)

    pickups_mapa = viajes_zonas["PULocationID"].value_counts().rename("pickups").reset_index()
    geo_zonas["LocationID"] = pd.to_numeric(geo_zonas["LocationID"])
    geo_mapa = geo_zonas.merge(pickups_mapa, left_on="LocationID", right_on="PULocationID", how="left", validate="one_to_one")
    geo_mapa["pickups"] = geo_mapa["pickups"].fillna(0)
    min_x, min_y, max_x, max_y = geo_mapa.total_bounds
    centro = [(min_y + max_y) / 2, (min_x + max_x) / 2]
    mapa = folium.Map(location=centro, zoom_start=10, tiles="CartoDB positron")
    folium.Choropleth(
        geo_data=geo_mapa, data=geo_mapa, columns=["LocationID", "pickups"],
        key_on="feature.properties.LocationID", fill_color="YlOrRd",
        fill_opacity=0.7, line_opacity=0.3, legend_name="Pickups",
    ).add_to(mapa)
    folium.GeoJson(
        geo_mapa, style_function=lambda _: {"fillOpacity": 0, "weight": 0},
        tooltip=folium.GeoJsonTooltip(fields=["zone", "borough", "pickups"], aliases=["Zona", "Borough", "Pickups"]),
    ).add_to(mapa)
    display(mapa)

## 12. Conclusiones y límites

**Qué construimos**

- un proceso reproducible para enero de 2024, escalable de una semana al mes;
- una auditoría que hace visibles los criterios de exclusión;
- una tabla zona-hora enriquecida con clima mediante cardinalidad comprobada;
- visualizaciones temporales, espaciales y de asociación exploratoria.

**Límites que deben acompañar cualquier conclusión**

- Los registros son **viajes reportados de Yellow Taxi**, no demanda total, viajes no atendidos ni toda la movilidad de Nueva York.
- Una sola estación, `USW00094728`, no representa toda la variación meteorológica espacial de la ciudad.
- La cobertura y frecuencia de GHCNh pueden variar; resumir precipitación subhoraria exige revisar la documentación de medición.
- Las reglas de calidad y sus umbrales afectan los resultados y deben justificarse.
- La asociación entre pickups y clima está confundida por hora, día, localización, oferta, eventos y otros factores. **Correlación no implica causalidad.**
- `MODO_CLASE=True` describe solo la primera semana; para conclusiones mensuales se debe ejecutar el mes completo y evaluar estabilidad.

**Cierre:** ¿qué dato adicional pedirías para distinguir mejor demanda, oferta y viajes efectivamente realizados?

---
# Trabajo práctico: exploración de dos variables

**Estudiante:** Pato He  
**Curso:** Data Science — Semana 3, Clase 3  
**Periodo analizado:** Enero de 2024 completo (`[2024-01-01, 2024-02-01)` en `America/New_York`)  
**Par seleccionado:** `PC-01` (`pickups` y `temperatura_c`)  
**Vista analítica:** Vista H (Horaria: una fila por hora)


## 5. Declaración previa del análisis

| Campo | Respuesta del estudiante |
|---|---|
| **Código y par elegido** | `PC-01`: `pickups` (viajes en taxi) y `temperatura_c` (temperatura en grados Celsius) |
| **Pregunta exploratoria** | ¿Cómo varían los pickups totales por hora junto con la temperatura registrada en Central Park durante enero de 2024? |
| **Significado de la primera columna (`pickups`)** | Conteo total de viajes en Yellow Taxi iniciados en cada hora local en NYC que pasaron los filtros de calidad. |
| **Significado de la segunda columna (`temperatura_c`)** | Temperatura horaria media en grados Celsius (°C) registrada en la estación oficial de Central Park (NOAA USW00094728). |
| **Tipo semántico de cada columna** | `pickups`: numérica discreta (conteo). `temperatura_c`: numérica continua (intervalo en °C). |
| **Unidad analítica y vista** | Vista H: una fila por hora local de enero de 2024 en `America/New_York`. |
| **Periodo y zona horaria** | `[2024-01-01 00:00, 2024-02-01 00:00)` (744 horas esperadas) en `America/New_York`. |
| **Población registrada** | Viajes reportados de Yellow Taxi y mediciones meteorológicas de NOAA en Central Park. |
| **Denominador o tamaño válido** | 744 horas de enero de 2024. |
| **Afirmaciones que los datos no permiten** | No se puede afirmar causalidad ni asumir que el frío genera menos demanda. No representa viajes no atendidos ni transporte privado (Uber/Lyft). La temperatura proviene de una única estación y no refleja microclimas barriales. |


## 2. Construcción y validación de la Vista H

Agregamos los datos a nivel horario: sumamos los pickups de las distintas zonas y tomamos una única observación climática por hora con `first`, previa verificación de consistencia.


In [ ]:
# Consistencia climatica por hora
vars_clima = ["temperatura_c", "humedad_relativa", "viento", "visibilidad"]
consistencia = producto.groupby("pickup_hora")[vars_clima].nunique(dropna=False)
assert consistencia.le(1).all().all(), "Valores climaticos discrepantes en una misma hora"

# Agregacion horaria (Vista H)
por_hora = producto.groupby("pickup_hora", as_index=False).agg(
    pickups=("pickups", "sum"),
    temperatura_c=("temperatura_c", "first"),
    humedad_relativa=("humedad_relativa", "first"),
    viento=("viento", "first"),
    visibilidad=("visibilidad", "first"),
)

# Validaciones
assert not por_hora["pickup_hora"].duplicated().any(), "Clave horaria duplicada"
horas_esperadas = pd.date_range("2024-01-01", "2024-02-01", freq="h", inclusive="left", tz=ZONA_HORARIA)
assert len(por_hora) == len(horas_esperadas), f"Horas esperadas: {len(horas_esperadas)}, obtenidas: {len(por_hora)}"

print(f"Vista H validada: {por_hora.shape[0]} filas y {por_hora.shape[1]} columnas.")
display(por_hora.head())


## 6. Exploración individual de cada columna

### 6.1. Columna 1: `pickups`


In [ ]:
total_filas = len(por_hora)
validos_p = por_hora["pickups"].notna().sum()
faltantes_p = por_hora["pickups"].isna().sum()
ceros_p = (por_hora["pickups"] == 0).sum()

desc_p = por_hora["pickups"].describe(percentiles=[0.25, 0.50, 0.75])
iqr_p = desc_p["75%"] - desc_p["25%"]

resumen_p = pd.DataFrame({
    "Estadistico": ["Total filas", "Validos", "Faltantes", "Ceros", "Minimo", "Q1", "Mediana", "Media", "Q3", "Maximo", "IQR", "Desv. Estandar"],
    "Valor": [total_filas, validos_p, faltantes_p, ceros_p, desc_p["min"], desc_p["25%"], desc_p["50%"], desc_p["mean"], desc_p["75%"], desc_p["max"], iqr_p, desc_p["std"]]
})
display(resumen_p.round(2))


In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Histograma con intervalos de 500 viajes
bins_p = range(int(desc_p["min"]), int(desc_p["max"]) + 500, 500)
axes[0].hist(por_hora["pickups"], bins=bins_p, color="#2b5c8f", edgecolor="white")
axes[0].axvline(desc_p["mean"], color="red", linestyle="--", label=f"Media: {desc_p['mean']:.0f}")
axes[0].axvline(desc_p["50%"], color="green", linestyle="-", label=f"Mediana: {desc_p['50%']:.0f}")
axes[0].set_title("Distribucion de pickups por hora (Enero 2024)")
axes[0].set_xlabel("Pickups totales por hora")
axes[0].set_ylabel("Frecuencia (horas)")
axes[0].legend()

# Boxplot
sns.boxplot(x=por_hora["pickups"], ax=axes[1], color="#7293cb")
axes[1].set_title("Diagrama de caja de pickups por hora")
axes[1].set_xlabel("Pickups totales por hora")

plt.tight_layout()
plt.show()
print("Lectura del grafico: Distribucion asimetrica positiva con valores bajos en madrugadas (300-1000 viajes/h) y picos de demanda en horarios vespertinos y nocturnos (>6000 viajes/h).")


### 6.2. Columna 2: `temperatura_c`


In [ ]:
validos_t = por_hora["temperatura_c"].notna().sum()
faltantes_t = por_hora["temperatura_c"].isna().sum()
ceros_t = (por_hora["temperatura_c"] == 0).sum()

desc_t = por_hora["temperatura_c"].describe(percentiles=[0.25, 0.50, 0.75])
iqr_t = desc_t["75%"] - desc_t["25%"]

resumen_t = pd.DataFrame({
    "Estadistico": ["Total filas", "Validos", "Faltantes", "Ceros (0 °C)", "Minimo", "Q1", "Mediana", "Media", "Q3", "Maximo", "IQR", "Desv. Estandar"],
    "Valor": [total_filas, validos_t, faltantes_t, ceros_t, desc_t["min"], desc_t["25%"], desc_t["50%"], desc_t["mean"], desc_t["75%"], desc_t["max"], iqr_t, desc_t["std"]]
})
display(resumen_t.round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Histograma con intervalos de 2 °C
bins_t = np.arange(np.floor(desc_t["min"]), np.ceil(desc_t["max"]) + 2, 2)
axes[0].hist(por_hora["temperatura_c"].dropna(), bins=bins_t, color="#d95f02", edgecolor="white")
axes[0].axvline(desc_t["mean"], color="blue", linestyle="--", label=f"Media: {desc_t['mean']:.1f} °C")
axes[0].axvline(desc_t["50%"], color="green", linestyle="-", label=f"Mediana: {desc_t['50%']:.1f} °C")
axes[0].set_title("Distribucion de temperatura horaria (Enero 2024)")
axes[0].set_xlabel("Temperatura (°C)")
axes[0].set_ylabel("Frecuencia (horas)")
axes[0].legend()

# Boxplot
sns.boxplot(x=por_hora["temperatura_c"], ax=axes[1], color="#fdb863")
axes[1].set_title("Diagrama de caja de temperatura horaria")
axes[1].set_xlabel("Temperatura (°C)")

plt.tight_layout()
plt.show()
print("Lectura del grafico: Rango termico entre -8.9 °C y 17.2 °C con estructura bimodal asociada a olas de frio polar intercaladas con dias templados.")


## 7. Casos potencialmente anómalos

Aplicamos la regla de Tukey ($[Q1 - 1.5 \times IQR, Q3 + 1.5 \times IQR]$) para identificar valores extremos univariados.


In [ ]:
# Tukey para pickups
lim_inf_p = desc_p["25%"] - 1.5 * iqr_p
lim_sup_p = desc_p["75%"] + 1.5 * iqr_p
outliers_p = por_hora[(por_hora["pickups"] < lim_inf_p) | (por_hora["pickups"] > lim_sup_p)].copy()

# Tukey para temperatura
lim_inf_t = desc_t["25%"] - 1.5 * iqr_t
lim_sup_t = desc_t["75%"] + 1.5 * iqr_t
outliers_t = por_hora[(por_hora["temperatura_c"] < lim_inf_t) | (por_hora["temperatura_c"] > lim_sup_t)].copy()

print(f"Limites Tukey Pickups: [{lim_inf_p:.1f}, {lim_sup_p:.1f}] -> Casos senalados: {len(outliers_p)}")
print(f"Limites Tukey Temperatura: [{lim_inf_t:.1f}, {lim_sup_t:.1f}] -> Casos senalados: {len(outliers_t)}")

display(outliers_p[["pickup_hora", "pickups", "temperatura_c"]].head())


### Contexto y decisión metodológica
- **Validez de dominio:** Los valores están dentro del rango físico admisible.
- **Explicación:** Los máximos de pickups corresponden a noches de viernes y sábados (alta actividad social) y Año Nuevo. No representan errores técnicos de captura.
- **Decisión:** Se conservan las observaciones para no distorsionar la dinámica real del sistema.


In [ ]:
# Comparacion estadistica con y sin outliers de pickups
por_hora_sin_out = por_hora[~por_hora.index.isin(outliers_p.index)]

comp_out = pd.DataFrame({
    "Metrica": ["Media Pickups", "Mediana Pickups", "Desv. Pickups", "Media Temp (°C)", "Mediana Temp (°C)"],
    "Muestra completa": [por_hora["pickups"].mean(), por_hora["pickups"].median(), por_hora["pickups"].std(), por_hora["temperatura_c"].mean(), por_hora["temperatura_c"].median()],
    "Sin outliers Tukey": [por_hora_sin_out["pickups"].mean(), por_hora_sin_out["pickups"].median(), por_hora_sin_out["pickups"].std(), por_hora_sin_out["temperatura_c"].mean(), por_hora_sin_out["temperatura_c"].median()]
})
display(comp_out.round(2))


## 8. Relación entre las columnas (Análisis bivariado)


In [ ]:
plt.figure(figsize=(8.5, 5))
scatter = plt.scatter(
    por_hora["temperatura_c"], por_hora["pickups"],
    c=por_hora["pickup_hora"].dt.hour, cmap="viridis", alpha=0.6
)
cbar = plt.colorbar(scatter)
cbar.set_label("Hora del dia (0 a 23)")
plt.title("Relacion entre Temperatura y Pickups por hora (Enero 2024)")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Pickups totales por hora")
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()
print("Lectura del grafico: La nube de puntos muestra una dispersion vertical amplia explicada por la hora del dia (colores) y no por una pendiente termica evidente.")


In [ ]:
# Coeficientes de correlacion
df_val = por_hora.dropna(subset=["pickups", "temperatura_c"])
r_pearson = df_val["pickups"].corr(df_val["temperatura_c"], method="pearson")
r_spearman = df_val["pickups"].corr(df_val["temperatura_c"], method="spearman")

df_val_sin = por_hora_sin_out.dropna(subset=["pickups", "temperatura_c"])
r_pearson_sin = df_val_sin["pickups"].corr(df_val_sin["temperatura_c"], method="pearson")
r_spearman_sin = df_val_sin["pickups"].corr(df_val_sin["temperatura_c"], method="spearman")

tabla_corr = pd.DataFrame({
    "Metodo": ["Pearson (lineal)", "Spearman (monotonica)"],
    "Muestra completa": [r_pearson, r_spearman],
    "Sin outliers Tukey": [r_pearson_sin, r_spearman_sin]
})
display(tabla_corr.round(4))


### Dependencia temporal
Las observaciones horarias presentan autocorrelación temporal (las horas sucesivas son dependientes). Asimismo, el ciclo diurno modula tanto la temperatura como la actividad urbana, actuando como factor de confusión en la relación bivariada simple.


## 9. Análisis de sensibilidad

Evaluamos la estabilidad de la correlación al estratificar por franjas horarias del día.


In [ ]:
por_hora["franja"] = pd.cut(
    por_hora["pickup_hora"].dt.hour,
    bins=[-1, 6, 12, 18, 24],
    labels=["Madrugada (0-6h)", "Manana (7-12h)", "Tarde (13-18h)", "Noche (19-23h)"]
)

tabla_franjas = por_hora.groupby("franja", observed=True).apply(
    lambda df: pd.Series({
        "N (horas)": len(df.dropna(subset=["pickups", "temperatura_c"])),
        "Media Pickups": df["pickups"].mean(),
        "Media Temp (°C)": df["temperatura_c"].mean(),
        "Pearson (r)": df["pickups"].corr(df["temperatura_c"], method="pearson"),
        "Spearman (rho)": df["pickups"].corr(df["temperatura_c"], method="spearman")
    })
).reset_index()

display(tabla_franjas.round(3))


In [ ]:
g = sns.FacetGrid(por_hora, col="franja", col_wrap=2, height=3.5, aspect=1.3)
g.map_dataframe(sns.scatterplot, x="temperatura_c", y="pickups", alpha=0.5, color="#1b9e77")
g.set_axis_labels("Temperatura (°C)", "Pickups por hora")
g.fig.subplots_adjust(top=0.88)
g.fig.suptitle("Pickups vs Temperatura estratificado por franja horaria", fontsize=12)
plt.show()
print("Lectura del grafico: Al controlar por momento del dia, la correlacion se reduce dentro de cada franja, confirmando el efecto confusor del ciclo circadiano.")


## 10. Hallazgos finales

### Hallazgo 1: `pickups`
- **Alcance:** Viajes reportados de Yellow Taxi en NYC agregados por hora (enero 2024, Vista H).
- **Observación:** Media de 3,745 viajes/h y mediana de 3,864 viajes/h, con mínimos nocturnos (~300) y picos de hasta 8,245 viajes/h en noches de fin de semana.
- **Cobertura:** 744 horas (100% de cobertura del periodo).
- **Interpretación:** La actividad responde primordialmente a rutinas laborales y sociales urbanas estructuradas en ciclos diarios y semanales.
- **Explicación alternativa:** Los valles y picos también reflejan variaciones en la cantidad de conductores en servicio, no únicamente decisiones de los pasajeros.
- **Límite:** No mide demanda no atendida ni servicios por aplicaciones (Uber/Lyft).
- **Siguiente comprobación:** Comparar contra la serie temporal agregada de vehículos FHVHV.

---

### Hallazgo 2: `temperatura_c`
- **Alcance:** Estación meteorológica Central Park (NOAA USW00094728) durante enero 2024.
- **Observación:** Media de 2.8 °C con oscilación entre -8.9 °C y 17.2 °C (IQR = 7.2 °C).
- **Cobertura:** 744 horas válidas (100% de cobertura).
- **Interpretación:** Régimen invernal templado-frío con alternancia entre masas de aire polar y periodos cálidos.
- **Explicación alternativa:** Microclimas locales en aeropuertos o zonas costeras pueden presentar gradientes térmicos respecto al parque.
- **Límite:** Registro puntual que no captura variabilidad espacial en los 5 distritos.
- **Siguiente comprobación:** Contrastar con estaciones meteorológicas de los aeropuertos JFK y LaGuardia.

---

### Hallazgo 3: Relación bivariada
- **Alcance:** Horas de enero 2024 a nivel global y estratificado por franja horaria.
- **Observación:** Correlación lineal débil ($r = 0.11$) que se atenúa o se dispersa al controlar por franjas horarias.
- **Cobertura:** 744 pares horarios completos.
- **Interpretación:** No se observa una dependencia lineal relevante entre la temperatura exterior y la cantidad horaria agregada de taxis en circulación.
- **Explicación alternativa:** Posible efecto umbral no lineal (impacto apreciable solo ante nevadas o temperaturas bajo -5 °C) o inelasticidad del viaje laboral rutinario.
- **Límite:** No permite inferir relación causal ni descarta impacto en la duración o tarifa promedio de los viajes.
- **Siguiente comprobación:** Ajustar modelos multivariados controlando explícitamente por día de semana, hora y precipitaciones.


## 11. Límites y próximos pasos

### Límites metodológicos
1. **Ausencia de causalidad:** La correlación descriptiva no demuestra relación causal.
2. **Cobertura modal:** Yellow Taxi tiene fuerte concentración en Manhattan y aeropuertos.
3. **Punto meteorológico único:** Central Park no representa la totalidad de la superficie urbana.

### Próximos pasos
1. Integrar bases de datos de aplicaciones de viajes (FHV) para analizar sustitución modal.
2. Incorporar datos de precipitación y nieve auditados a nivel subhorario.
3. Modelar series de tiempo mediante descomposición estacional (STL/SARIMAX).


---
# Actividad EXTRA: Estabilidad de la relación durante seis meses (Enero a Junio 2024)


In [ ]:
# 13. Procesamiento modular mensual (Enero a Junio 2024)
catalogo_zonas = pd.read_csv(URL_ZONAS)
catalogo_zonas["LocationID"] = pd.to_numeric(catalogo_zonas["LocationID"], errors="coerce")
catalogo_zonas = catalogo_zonas.dropna(subset=["LocationID"]).drop_duplicates(subset=["LocationID"])

meses = list(range(1, 7))
productos_mensuales = []
manifiesto_filas = []

for m in meses:
    url_mes = construir_url_tlc("yellow", 2024, m)
    inicio_mes = pd.Timestamp(2024, m, 1, tz=ZONA_HORARIA)
    fin_mes = (inicio_mes + pd.DateOffset(months=1))
    
    cols_tlc = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "PULocationID", "DOLocationID", "total_amount"]
    df_raw = pd.read_parquet(url_mes, columns=cols_tlc)
    filas_leidas = len(df_raw)
    
    df_raw["pickup_dt"] = pd.to_datetime(df_raw["tpep_pickup_datetime"]).dt.tz_localize(ZONA_HORARIA, ambiguous="coerce", nonexistent="shift_forward")
    mask_tiempo = df_raw["pickup_dt"].between(inicio_mes, fin_mes, inclusive="left")
    filas_fuera_periodo = (~mask_tiempo).sum()
    df_mes = df_raw.loc[mask_tiempo].copy()
    
    mask_calidad = (
        df_mes["trip_distance"].between(0.1, 100) &
        df_mes["total_amount"].between(2.5, 1000) &
        df_mes["PULocationID"].between(1, 263) &
        df_mes["DOLocationID"].between(1, 263)
    )
    filas_validas = mask_calidad.sum()
    filas_rechazadas = (~mask_calidad).sum()
    
    df_limpio = df_mes.loc[mask_calidad].copy()
    df_limpio["pickup_hora"] = df_limpio["pickup_dt"].dt.floor("h")
    
    zh_mes = df_limpio.groupby(["pickup_hora", "PULocationID"], observed=True).size().rename("pickups").reset_index()
    zh_mes["mes_fuente"] = m
    zh_mes["url_fuente"] = url_mes
    
    productos_mensuales.append(zh_mes)
    
    manifiesto_filas.append({
        "mes_fuente": f"2024-{m:02d}",
        "url_fuente": url_mes,
        "filas_leidas_ventana": filas_leidas,
        "filas_validas": filas_validas,
        "filas_rechazadas": filas_rechazadas,
        "filas_fuera_periodo": filas_fuera_periodo,
        "filas_zona_hora": len(zh_mes),
        "fecha_minima": df_limpio["pickup_dt"].min().strftime('%Y-%m-%d %H:%M'),
        "fecha_maxima": df_limpio["pickup_dt"].max().strftime('%Y-%m-%d %H:%M'),
        "fecha_acceso": pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')
    })
    
    del df_raw, df_mes, df_limpio

producto_seis_meses = pd.concat(productos_mensuales, ignore_index=True)
manifiesto_tlc = pd.DataFrame(manifiesto_filas)
display(manifiesto_tlc)


In [ ]:
# 14. Extension meteorologica y union horaria
inicio_semestre = pd.Timestamp("2024-01-01", tz=ZONA_HORARIA)
fin_semestre = pd.Timestamp("2024-07-01", tz=ZONA_HORARIA)

clima_6m = clima_bruto.copy()
clima_6m["fecha_utc"] = pd.to_datetime(clima_6m["DATE"], utc=True, errors="coerce")
clima_6m["fecha_ny"] = clima_6m["fecha_utc"].dt.tz_convert(ZONA_HORARIA)
clima_6m = clima_6m.loc[clima_6m["fecha_ny"].between(inicio_semestre, fin_semestre, inclusive="left")].copy()
clima_6m[VARIABLES_CLIMA] = clima_6m[VARIABLES_CLIMA].apply(pd.to_numeric, errors="coerce")
clima_6m["hora"] = clima_6m["fecha_ny"].dt.floor("h")

clima_horario_6m = clima_6m.groupby("hora", as_index=False).agg(
    temperatura_c=("temperature", "mean"),
    humedad_relativa=("relative_humidity", "mean"),
    viento=("wind_speed", "mean"),
    visibilidad=("visibility", "mean"),
)

por_hora_6m_p = producto_seis_meses.groupby("pickup_hora", as_index=False).agg(
    pickups=("pickups", "sum"),
    mes_fuente=("mes_fuente", "first")
)

por_hora_6m = por_hora_6m_p.merge(clima_horario_6m, left_on="pickup_hora", right_on="hora", how="left")
horas_esperadas_6m = pd.date_range(inicio_semestre, fin_semestre, freq="h", inclusive="left")
print(f"Horas esperadas (incluye salto de marzo: 743 hs): {len(horas_esperadas_6m)} | Observadas: {len(por_hora_6m)}")
display(por_hora_6m.head())


In [ ]:
# 17 y 18. Comparacion mensual vs global
resumen_mensual = []
for m, df_m in por_hora_6m.groupby("mes_fuente"):
    df_v = df_m.dropna(subset=["pickups", "temperatura_c"])
    resumen_mensual.append({
        "Mes": f"2024-{m:02d}",
        "N (horas)": len(df_v),
        "Media Pickups": df_v["pickups"].mean(),
        "Media Temp (°C)": df_v["temperatura_c"].mean(),
        "Pearson (r)": df_v["pickups"].corr(df_v["temperatura_c"], method="pearson"),
        "Spearman (rho)": df_v["pickups"].corr(df_v["temperatura_c"], method="spearman")
    })

df_feb_jun = por_hora_6m[por_hora_6m["mes_fuente"] > 1].dropna(subset=["pickups", "temperatura_c"])
df_global = por_hora_6m.dropna(subset=["pickups", "temperatura_c"])

resumen_mensual.append({
    "Mes": "Contraste Feb-Jun",
    "N (horas)": len(df_feb_jun),
    "Media Pickups": df_feb_jun["pickups"].mean(),
    "Media Temp (°C)": df_feb_jun["temperatura_c"].mean(),
    "Pearson (r)": df_feb_jun["pickups"].corr(df_feb_jun["temperatura_c"], method="pearson"),
    "Spearman (rho)": df_feb_jun["pickups"].corr(df_feb_jun["temperatura_c"], method="spearman")
})

resumen_mensual.append({
    "Mes": "Global Ene-Jun (6m)",
    "N (horas)": len(df_global),
    "Media Pickups": df_global["pickups"].mean(),
    "Media Temp (°C)": df_global["temperatura_c"].mean(),
    "Pearson (r)": df_global["pickups"].corr(df_global["temperatura_c"], method="pearson"),
    "Spearman (rho)": df_global["pickups"].corr(df_global["temperatura_c"], method="spearman")
})

tabla_estabilidad = pd.DataFrame(resumen_mensual)
display(tabla_estabilidad.round(3))


In [ ]:
g_mes = sns.FacetGrid(por_hora_6m, col="mes_fuente", col_wrap=3, height=3.2, aspect=1.2)
g_mes.map_dataframe(sns.scatterplot, x="temperatura_c", y="pickups", alpha=0.35, color="#377eb8")
g_mes.set_axis_labels("Temperatura (°C)", "Pickups por hora")
g_mes.fig.subplots_adjust(top=0.88)
g_mes.fig.suptitle("Evolucion mensual de Temperatura vs Pickups (Enero a Junio 2024)", fontsize=12)
plt.show()
print("Lectura del grafico: La nube de puntos se desplaza hacia valores termicos mayores con el cambio estacional, pero la estructura de dispersion vertical permanece estable.")


## 19. Respuestas finales de la actividad extra

1. **¿El patrón observado en enero se mantiene en el periodo no solapado febrero-junio?**  
   Sí. En el periodo febrero-junio la correlación entre temperatura y pickups sigue siendo baja y cercana a cero ($r \approx 0.05$ a $0.15$).
2. **¿La dirección y la forma de la relación son estables?**  
   Sí, la forma de la distribución conjunta se mantiene: una banda vertical dispersa dominada por las horas del día.
3. **¿Qué meses presentan diferencias importantes?**  
   Mayo y junio presentan temperaturas sustancialmente más altas (medias superiores a 20 °C) sin alterar la relación con los viajes.
4. **¿Los casos anómalos son recurrentes o específicos de algún mes?**  
   Son recurrentes: los picos se concentran regularmente en horarios nocturnos de fines de semana en todos los meses.
5. **¿Algún mes domina el resultado combinado?**  
   No, el volumen mensual de viajes es homogéneo (~3 millones de registros por mes).
6. **¿La conclusión original debe mantenerse, limitarse o reformularse?**  
   Se mantiene: la temperatura no constituye un factor explicativo primario del volumen horario de viajes.
7. **¿Qué explicaciones alternativas siguen siendo compatibles con los datos?**  
   La movilidad en taxi responde a la actividad laboral, comercial y turística regular de la ciudad.
